# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ishkaklotar03/ML-Internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# W01: Research Question and Provisional Lane

## 1) My lane and why

**Provisional lane:** Lane 2, Refresh / Content Opportunity Scoring.

**Why this lane:** The starter dataset and the starter pipeline are built for this lane, so I can start from a transparent baseline score and test whether a learned ranking does better. The output is a ranked list of pages to review first, which matches a real constraint: reviewers have limited time. The lane also leaves room to move later toward stronger future-window labels using the warehouse daily table.

**One-paragraph plan:** Over the next 7 weeks I will (1) define a safe data contract on the starter data, (2) audit which observable signals (impressions, position, CTR, sessions, age, freshness) carry information, (3) build a simple rule-based baseline score with reason codes, (4) test whether a model (logistic regression, then tree-based) ranks better using client-holdout validation, (5) check the top of the ranking by hand, and (6) move to a future-window label (prior 90 days of features, next 30 days outcome) using the warehouse release. I can confirm or change this lane until the end of Week 4.## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*


In [5]:
print("Lane 2: Refresh / Content Opportunity Scoring")

Lane 2: Refresh / Content Opportunity Scoring


## 2) The question: decision, action, and cost of a wrong call

**Search question:** Which already-published pages with real search demand should a content reviewer look at first because their visibility appears to be weakening?

**Unit of analysis:** One row = one content page (`content_id`) within a client, measured over a 90-day window. I will not compare pages across clients without checking client-level differences.

**Output:** A ranked review queue. Each page gets a score, a short reason code (for example `declining_with_demand`, `stale_visible_page`, `low_ctr_visible_page`) and a suggested review action.

**Decision it improves:** Which pages a content team reviews first when it can only review a limited number (for example 20 to 50 pages) in a given cycle.

**Action someone could take:** Open the top-ranked pages and decide whether to refresh the content, rewrite the title and meta description, improve the page structure, or just keep monitoring.

**Cost of a wrong recommendation:**
- *False positive* (a page ranked high that is actually fine): reviewer time is wasted, and an unnecessary edit could even disturb a page that was performing well.
- *False negative* (a page that is really declining but ranked low): the decline continues unnoticed and the traffic loss grows.
- Because review capacity is limited, false positives at the top of the list matter most, so I will judge the ranking with precision@K (for example precision@50) instead of generic accuracy.

**Why data or ML can help, and why it is not just "train a model":**
- A fixed rule (for example "old page with many impressions") is simple but treats every signal the same for every page. Data can show which signals actually go together with declining visibility.
- ML might rank pages better than a fixed rule, but only if it is checked honestly. On the starter data a rule baseline will be my reference point, and a model has to beat it under client-holdout validation.
- The starter label (`trend_direction == "down"`) is only a proxy computed from the current window, not a future outcome. So the real work is defining the question, the label, the window, and the thresholds correctly, not choosing an algorithm.
- A high score means "review this page first", not "this page will recover if edited". The data cannot prove that a refresh causes recovery.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

Starter file: data/raw/content_refresh_anonymized.csv

In [7]:
import os
import pandas as pd
from pathlib import Path

# Colab ma repo na hoy to clone karo
if not Path("ML-Internship").exists() and not Path("data/raw/content_refresh_anonymized.csv").exists():
    !git clone https://github.com/ishkaklotar03/ML-Internship.git

candidates = [
    Path("ML-Internship/data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("data/raw/content_refresh_anonymized.csv"),
]
path = next((p for p in candidates if p.exists()), None)

if path is None:
    print("CSV malyu nahi. Files:", os.listdir("."))
    raise FileNotFoundError("content_refresh_anonymized.csv not found")

df = pd.read_csv(path)

n_rows = len(df)
n_clients = df["client_id"].nunique()
print(f"Number 1 - rows: {n_rows:,} | clients: {n_clients:,}")

elig = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]
print(f"Eligible rows: {len(elig):,} ({len(elig)/n_rows:.1%})")

down_share = (elig["trend_direction"] == "down").mean()
print(f"Number 2 - share of eligible pages with trend 'down': {down_share:.1%}")

imp = elig["impressions_90d"].sort_values(ascending=False)
top10_share = imp.head(int(len(imp) * 0.10)).sum() / imp.sum()
print(f"Number 3 - top 10% of pages hold {top10_share:.1%} of impressions_90d")

Number 1 - rows: 30,000 | clients: 32
Eligible rows: 30,000 (100.0%)
Number 2 - share of eligible pages with trend 'down': 54.2%
Number 3 - top 10% of pages hold 70.2% of impressions_90d


## 4) Careful words: what I can and cannot claim

**What I can claim (if the data supports it):**
- "In this starter slice, pages with X and Y are more often labelled as trending down."
- "A ranking built from observable signals placed more pages with a down trend in its top K than a simple rule did, under client-holdout validation."
- "These pages are good candidates for human review first."

**What I cannot claim:**
- That a refresh *caused* any recovery. This data is observational and I have not run an experiment or causal design.
- That I found a Google ranking factor or algorithm behaviour.
- That a "down" trend is a real, lasting decline. The starter label is a proxy computed from the current window, and it could also reflect seasonality, consolidation into another page, or noise.
- That results from a ~30,000-row starter slice carry over to the full warehouse. They have to be re-tested there.
- Anything about AI citations or AI search rankings.

**Language rule I will follow:** use "observed", "associated with", "suggests", "candidate for review". Avoid "proves", "guarantees", "will recover", "the best page to fix".

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Careful-claims check: label is a proxy, not a future outcome
print("Label used:", "trend_direction == 'down' (proxy, current window)")
print("Causal claims made: none")

Label used: trend_direction == 'down' (proxy, current window)
Causal claims made: none


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.